In [2]:
import numpy as np
import pandas as pd
import os
import librosa
import tkinter as tk
from tkinter import filedialog, messagebox, Label, Button
from tensorflow.keras.models import load_model

# Load the trained model
model = load_model('E:/Users/Sumit/Downloads/UrbanSound8K/UrbanSound8K/saved_models_CRNN/audio_classification_CRNN.keras')

# List of class names in the order corresponding to model's output layer
class_names = ["air_conditioner", "car_horn", "children_playing", "dog_bark", "drilling", 
               "engine_idling", "gun_shot", "jackhammer", "siren", "street_music"]

def features_extractor(file_name):
    audio, sample_rate = librosa.load(file_name, res_type='kaiser_fast')
    mfccs_features = librosa.feature.mfcc(y=audio, sr=sample_rate, n_mfcc=40)
    mfccs_scaled_features = np.mean(mfccs_features.T, axis=0)
    return mfccs_scaled_features

# Define GUI Application
class AudioClassifierApp:
    def __init__(self, root):
        self.root = root
        self.root.title("Acoustic Scene Classification")
        self.root.geometry("400x200")

        # Label
        self.label = Label(root, text="Acoustic Scene Classifier using CRNN", font=("Helvetica", 14))
        self.label.pack(pady=10)

        # Buttons for upload and predict
        self.upload_button = Button(root, text="Upload Audio File", command=self.upload_file)
        self.upload_button.pack(pady=5)
        
        self.predict_button = Button(root, text="Predict Class", command=self.predict_class, state="disabled")
        self.predict_button.pack(pady=5)

        # Result label
        self.result_label = Label(root, text="", font=("Helvetica", 12), fg="blue")
        self.result_label.pack(pady=10)

        # Initialize variables
        self.file_path = None

    def upload_file(self):
        self.file_path = filedialog.askopenfilename(filetypes=[("Audio Files", "*.wav")])
        if self.file_path:
            self.result_label.config(text="File loaded successfully!")
            self.predict_button.config(state="normal")
        else:
            self.result_label.config(text="No file selected.")

    def predict_class(self):
        if self.file_path:
            # Extract features from the audio file
            feature = features_extractor(self.file_path)
            feature = feature.reshape(1, 1, feature.shape[0], 1)

            # Predict class
            predictions = model.predict(feature)
            predicted_class_index = np.argmax(predictions, axis=-1).flatten()[0]
            prediction_label = class_names[predicted_class_index]

            # Display the result
            self.result_label.config(text=f"Predicted class: {prediction_label}")
        else:
            messagebox.showwarning("Warning", "Please upload an audio file first.")

# Main
if __name__ == "__main__":
    root = tk.Tk()
    app = AudioClassifierApp(root)
    root.mainloop()


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 358ms/step
